# Runtime Plot (benchmark_runs)

Reads JSON files saved by `--save-metrics` and plots CPU vs GPU runtime by repeat factor.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_dir = Path.cwd()
if not (base_dir / 'benchmark_runs').exists():
    candidate = (Path.cwd() / 'notebooks' / 'profiling').resolve()
    if (candidate / 'benchmark_runs').exists():
        base_dir = candidate

runs_dir = base_dir / 'benchmark_runs'
print('runs_dir:', runs_dir)

records = []
for p in sorted(runs_dir.glob('*.json')):
    data = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
    data['source_file'] = p.name
    records.append(data)

if not records:
    raise RuntimeError('No metrics JSON found. Run profiling with --save-metrics first.')

df = pd.DataFrame(records)
df['repeat_factor'] = pd.to_numeric(df['repeat_factor'])
df['elapsed_seconds'] = pd.to_numeric(df['elapsed_seconds'])
df['mode_label'] = np.where(df['use_gpu'], 'GPU bitset', 'CPU bitset')
df[['mode_label', 'repeat_factor', 'elapsed_seconds', 'rule_count', 'source_file']].head()


In [ ]:
summary = (
    df.groupby(['mode_label', 'repeat_factor'], as_index=False)
      .agg(
          runs=('elapsed_seconds', 'count'),
          mean_s=('elapsed_seconds', 'mean'),
          median_s=('elapsed_seconds', 'median'),
          std_s=('elapsed_seconds', lambda x: float(x.std(ddof=1)) if len(x) > 1 else 0.0),
          rules_min=('rule_count', 'min'),
          rules_max=('rule_count', 'max'),
      )
      .sort_values(['mode_label', 'repeat_factor'])
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for mode, g in summary.groupby('mode_label'):
    ax.errorbar(
        g['repeat_factor'],
        g['mean_s'],
        yerr=g['std_s'],
        marker='o',
        linewidth=2,
        capsize=4,
        label=mode,
    )
    for _, row in g.iterrows():
        ax.annotate(
            f"{row['mean_s']:.3f} +/- {row['std_s']:.3f}",
            xy=(row['repeat_factor'], row['mean_s'] + row['std_s']),
            xytext=(0, 6),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=8,
        )

ax.set_xlabel('Repeat factor')
ax.set_ylabel('Runtime mean (s)')
ax.set_title('CPU vs GPU bitset runtime by repeat factor')
ax.grid(alpha=0.25)
ax.legend()

out_png = base_dir / 'runtime_vs_repeat_bitset_with_std.png'
plt.tight_layout()
plt.savefig(out_png, dpi=300, format='png')
print('Saved:', out_png)
plt.show()
